### 1-Library

In [1]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile, BadZipFile
import re

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"

print(f"Projeto: {PROJECT_ROOT}")
print(f"Bronze: {BRONZE_DIR}")

Projeto: /home/akel/PycharmProjects/ENEM
Bronze: /home/akel/PycharmProjects/ENEM/data/bronze


### 2. identificar o ano

In [2]:
def extrair_ano(nome_arquivo: str) -> int:
    """
    Extrai o ano do nome de um arquivo como:
    microdados_enem_1998.zip
    """
    resultado = re.search(r"(19|20)\d{2}", nome_arquivo)

    if resultado is None:
        raise ValueError(
            f"Não foi possível identificar o ano em: {nome_arquivo}"
        )

    return int(resultado.group())
    
extrair_ano("microdados_enem_1998.zip")

1998

### 3. classificar o papel de cada CSV 

In [3]:
def classificar_csv(caminho_interno: str) -> str:
    """
    Classifica o tipo de CSV encontrado dentro do ZIP.
    """
    nome = PurePosixPath(caminho_interno).name.upper()

    if "PARTICIPANTES" in nome:
        return "participantes"

    if "RESULTADOS" in nome:
        return "resultados"

    if "MICRODADOS_ENEM" in nome:
        return "microdados"

    if "ITENS_PROVA" in nome:
        return "itens_prova"

    if nome.startswith("QUEST_"):
        return "questionario"

    return "csv_desconhecido"

print(classificar_csv("DADOS/MICRODADOS_ENEM_1998.csv"))
print(classificar_csv("microdados_enem_2024/DADOS/PARTICIPANTES_2024.csv"))
print(classificar_csv("microdados_enem_2024/DADOS/RESULTADOS_2024.csv"))

microdados
participantes
resultados


### 4.mapear todos os ZIPs

In [4]:
registros = []
erros = []

arquivos_zip = sorted(BRONZE_DIR.glob("microdados_enem_*.zip"))

for caminho_zip in arquivos_zip:
    ano = extrair_ano(caminho_zip.name)  #<--usando função

    try:
        with ZipFile(caminho_zip, mode="r") as arquivo_zip:
            for info in arquivo_zip.infolist():

                if info.is_dir():
                    continue

                if not info.filename.lower().endswith(".csv"):
                    continue
                tamanho_mb = info.file_size / 1024**2

                #iterado por ano
                registros.append(
                    {
                        "ano": ano,
                        "arquivo_zip": caminho_zip.name,
                        "caminho_interno": info.filename,
                        "nome_csv": PurePosixPath(info.filename).name,
                        "tipo": classificar_csv(info.filename),    #<--usando função caminho_interno
                        "tamanho_mb": tamanho_mb,
                    }
                )

    except BadZipFile as erro:
        erros.append(
            {
                "ano": ano,
                "arquivo_zip": caminho_zip.name,
                "erro": str(erro),
            }
        )

mapa_csv = pd.DataFrame(registros) 
erros_zip = pd.DataFrame(erros)

print(f"ZIPs analisados: {len(arquivos_zip)}")
print(f"CSVs encontrados: {len(mapa_csv)}")
print(f"ZIPs com erro: {len(erros_zip)}")

ZIPs analisados: 28
CSVs encontrados: 48
ZIPs com erro: 0


### 5.visualizar o mapeamento completo

In [5]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 150)

mapa_csv.sort_values(
    by=["ano", "tipo", "nome_csv"]
).reset_index(drop=True)

,ano,arquivo_zip,caminho_interno,nome_csv,tipo,tamanho_mb
0,1998,microdados_enem_1998.zip,DADOS/MICRODADOS_ENEM_1998.csv,MICRODADOS_ENEM_1998.csv,microdados,60.144619
1,1999,microdados_enem_1999.zip,DADOS/MICRODADOS_ENEM_1999.csv,MICRODADOS_ENEM_1999.csv,microdados,138.045591
2,2000,microdados_enem_2000.zip,DADOS/MICRODADOS_ENEM_2000.csv,MICRODADOS_ENEM_2000.csv,microdados,165.594666
3,2001,microdados_enem_2001.zip,DADOS/MICRODADOS_ENEM_2001.csv,MICRODADOS_ENEM_2001.csv,microdados,1026.466652
4,2002,microdados_enem_2002.zip,DADOS/MICRODADOS_ENEM_2002.csv,MICRODADOS_ENEM_2002.csv,microdados,1014.886230
5,2003,microdados_enem_2003.zip,DADOS/MICRODADOS_ENEM_2003.csv,MICRODADOS_ENEM_2003.csv,microdados,950.585272
6,2004,microdados_enem_2004.zip,Dados/MICRODADOS_ENEM_2004.csv,MICRODADOS_ENEM_2004.csv,microdados,806.656661
7,2005,microdados_enem_2005.zip,DADOS/MICRODADOS_ENEM_2005.csv,MICRODADOS_ENEM_2005.csv,microdados,1662.940553
8,2006,microdados_enem_2006.zip,DADOS/MICRODADOS_ENEM_2006.csv,MICRODADOS_ENEM_2006.csv,microdados,2080.403831
9,2007,microdados_enem_2007.zip,DADOS/MICRODADOS_ENEM_2007.csv,MICRODADOS_ENEM_2007.csv,microdados,1978.589364


### 6. classificar o layout de cada ano (reorganizar o dataframe)

In [6]:
# 1.função de classificação
def classificar_layout(tipos_csv) -> str:
    """
    Classifica a organização dos arquivos CSV de uma edição.
    """
    tipos_csv = set(tipos_csv)

    if {"participantes", "resultados"}.issubset(tipos_csv):
        return "arquivos_separados"

    if "microdados" in tipos_csv:
        return "arquivo_unico"

    return "revisar"

# 2. agrupar os registros por ano
grupos_por_ano = mapa_csv.groupby("ano")   

# 3.contar os CSVs de cada ano
quantidade_csv = (grupos_por_ano.size().reset_index(name="quantidade_csv"))

# AGRUPAR p/ ANO
# 4.Agrupar os nomes dos arquivos 
arquivos_por_ano = (
    grupos_por_ano["nome_csv"]
    .apply(lambda nomes: " | ".join(sorted(nomes)))
    .reset_index(name="arquivos")
)

# 5.Agrupar tipo de arquivo 
tipos_por_ano = (
    grupos_por_ano["tipo"]
    .apply(lambda tipos: sorted(set(tipos)))
    .reset_index(name="tipos_lista")
)


# 6.calcular o tamanho total dos CSV 
tamanho_por_ano = (
    grupos_por_ano["tamanho_mb"]
    .sum()
    .reset_index(name="tamanho_total_mb")
)

# JUNTAR RESULTADOS
# QUANTIDADE
resumo_layout = quantidade_csv.merge(
    arquivos_por_ano,
    on="ano",
    how="left",)

# TIPO
resumo_layout = resumo_layout.merge(
    tipos_por_ano,
    on="ano",
    how="left",)

# TAMANHO
resumo_layout = resumo_layout.merge(
    tamanho_por_ano,
    on="ano",
    how="left",)

#aplicar a função de classificacao
resumo_layout["layout_detectado"] = (
    resumo_layout["tipos_lista"]
    .apply(classificar_layout)
)

resumo_layout["tipo"] = (
    resumo_layout["tipos_lista"]
    .apply(lambda tipos: " | ".join(tipos))
)

resumo_layout[
    [
        "ano",
        "quantidade_csv",
        "arquivos",
        "tipo",
        "tamanho_total_mb",
        "layout_detectado",
    ]
]

,ano,quantidade_csv,arquivos,tipo,tamanho_total_mb,layout_detectado
0,1998,1,MICRODADOS_ENEM_1998.csv,microdados,60.144619,arquivo_unico
1,1999,1,MICRODADOS_ENEM_1999.csv,microdados,138.045591,arquivo_unico
2,2000,1,MICRODADOS_ENEM_2000.csv,microdados,165.594666,arquivo_unico
3,2001,1,MICRODADOS_ENEM_2001.csv,microdados,1026.466652,arquivo_unico
4,2002,1,MICRODADOS_ENEM_2002.csv,microdados,1014.886230,arquivo_unico
5,2003,1,MICRODADOS_ENEM_2003.csv,microdados,950.585272,arquivo_unico
6,2004,1,MICRODADOS_ENEM_2004.csv,microdados,806.656661,arquivo_unico
7,2005,1,MICRODADOS_ENEM_2005.csv,microdados,1662.940553,arquivo_unico
8,2006,1,MICRODADOS_ENEM_2006.csv,microdados,2080.403831,arquivo_unico
9,2007,1,MICRODADOS_ENEM_2007.csv,microdados,1978.589364,arquivo_unico


### 7.comparar com o layout esperado

In [7]:
def layout_esperado(ano: int) -> str:
    if ano in {2024, 2025}:
        return "arquivos_separados"

    return "arquivo_unico"


resumo_layout["layout_esperado"] = (
    resumo_layout["ano"].apply(layout_esperado)
)

resumo_layout["status"] = resumo_layout.apply(
    lambda linha: (
        "OK"
        if linha["layout_detectado"] == linha["layout_esperado"]
        else "REVISAR"
    ),
    axis=1,
)

resumo_layout[
    [
        "ano",
        "quantidade_csv",
        "tipo",
        "layout_detectado",
        "layout_esperado",
        "status",
    ]
]

,ano,quantidade_csv,tipo,layout_detectado,layout_esperado,status
0,1998,1,microdados,arquivo_unico,arquivo_unico,OK
1,1999,1,microdados,arquivo_unico,arquivo_unico,OK
2,2000,1,microdados,arquivo_unico,arquivo_unico,OK
3,2001,1,microdados,arquivo_unico,arquivo_unico,OK
4,2002,1,microdados,arquivo_unico,arquivo_unico,OK
5,2003,1,microdados,arquivo_unico,arquivo_unico,OK
6,2004,1,microdados,arquivo_unico,arquivo_unico,OK
7,2005,1,microdados,arquivo_unico,arquivo_unico,OK
8,2006,1,microdados,arquivo_unico,arquivo_unico,OK
9,2007,1,microdados,arquivo_unico,arquivo_unico,OK


### 8. verficação final

In [8]:
divergencias = resumo_layout[
    resumo_layout["status"] != "OK"
]

if divergencias.empty:
    print("Todos os anos apresentam o layout esperado.")
else:
    print("Foram encontradas edições que precisam de revisão:")
    display(divergencias)

Todos os anos apresentam o layout esperado.


In [9]:
if erros_zip.empty:
    print("Nenhum erro de abertura foi identificado nos ZIPs.")
else:
    display(erros_zip)

Nenhum erro de abertura foi identificado nos ZIPs.


In [10]:

csvs_desconhecidos = mapa_csv[
    mapa_csv["tipo"] == "csv_desconhecido"
]

print(
    f"Total de CSVs desconhecidos: "
    f"{len(csvs_desconhecidos)}"
)

csvs_desconhecidos

Total de CSVs desconhecidos: 0


,ano,arquivo_zip,caminho_interno,nome_csv,tipo,tamanho_mb
